In [ ]:
# For tips on running notebooks in Google Colab, see
# https://pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


//自增笔记📒：

torch.nn命名空间中提供了构建自定义神经网络所需的所有基础组件。

PyTorch中的每个模块都继承自nn.Module。
神经网络本身也是一个模块，可以包含其他模块。



In [1]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


//自增笔记📒：

加载accelerator模块，假如有可用的加速设备，就用；没有就自动适用CPU，便于跨平台训练

In [3]:
device = torch.cuda.current_accelerator().type if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [ ]:
class NeuralNetwork(nn.Module):#继承自nn.Module的子类，用于定义我们自己的神经网络
    def __init__(self):#通过__init__来进行初始化
        super().__init__()#调用父类__init__方法，让父类完成自己的初始化工作
        self.flatten = nn.Flatten()#开始初始化自己的独有内容
        self.linear_relu_stack = nn.Sequential(#典型的线性层➕ReLU激活函数
            nn.Linear(28*28, 512),#这些维度都可以在最后一个单元格的参数维度输出中找到相应值
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)#把输入数据展平为一维向量
        logits = self.linear_relu_stack(x)#传入堆叠的线性层+激活层，计算原始输出结果
        return logits#未经概率激活的原始值

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [5]:
model = NeuralNetwork().to(device)#移到设备上，并打印结构
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


//自增笔记📒：

用模型时，直接把数据传给模型实例model(input),不要手动写model.forward(input)

前者会自动执行forward，还会同时跑一些后台必要操作（如钩子，注册，模式管理），而直接调用forward会跳过这些容易出错

- 模型输出什么？
- 样本 × 类别原始分数：
  
  dim=0:每一行对应一个样本的是个原始预测值
  
  dim=1:每一列对应一个类别相应的得分

- 原始输出值不是概率，logits要经过softmax（概率激活）后才会变成0到1之间，总和为1到类别概率

In [6]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)#创建softmax层后在第一维做归一化
y_pred = pred_probab.argmax(1)#取一行中概率预测值最大的索引位置作为标签【注意不是最大值本身】，(1)：dim=1，指定按行计算
print(f"Predicted class: {y_pred}")

Predicted class: tensor([0])


------------------------------------------------------------------------


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [7]:
input_image = torch.rand(3,28,28)#批大小=3，每个size=28X28
print(input_image.size())

torch.Size([3, 28, 28])


nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [8]:
flatten = nn.Flatten()
flat_image = flatten(input_image)#展平原来的28X28维，flatten默认从第一维开始展平到最后一维，保留第零维
print(flat_image.size())

torch.Size([3, 784])


nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [ ]:
layer1 = nn.Linear(in_features=28*28, out_features=20)#创建一个全连接层，input为28*28，输出特征数是20；内部权重矩阵【20，784】；偏置矩阵【20】。Y=x*W^T+b
hidden1 = layer1(flat_image)#把扁平化的图像数据输入第一层隐藏层，计算得到第一层输出特征
print(hidden1.size())

torch.Size([3, 20])


nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [10]:
print(f"Before ReLU: {hidden1}\n\n")#ReLU是引入非线性因素的一种方式，但也还存在其他引入非线性内容的方式
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")#ReLU(x)=max(x,0)

Before ReLU: tensor([[ 4.9756e-01,  4.4916e-01, -7.5693e-01,  1.6183e-01, -2.0276e-02,
         -5.8690e-02,  3.4835e-01, -2.6889e-01, -7.8561e-02,  3.3159e-01,
         -1.0759e-01, -2.4530e-01,  6.0231e-02, -4.6829e-01,  2.4670e-01,
         -3.1843e-02, -5.0648e-02, -6.7883e-02,  2.0721e-01,  6.4557e-02],
        [ 2.4652e-01,  4.2856e-01, -6.1009e-01,  2.2331e-01, -2.7803e-01,
         -3.8848e-01,  3.8622e-01, -2.0483e-01, -2.8230e-01,  6.6703e-01,
          2.9655e-02, -5.6782e-01, -1.5720e-01,  2.8629e-02,  1.0944e-01,
         -1.3461e-01, -7.5617e-04, -2.2658e-01, -2.3052e-01, -9.9086e-02],
        [ 3.6886e-01,  2.6032e-01, -7.7855e-01,  4.5380e-01,  8.1398e-02,
         -1.2901e-01,  3.6491e-01, -1.8597e-01, -3.5400e-01,  5.5474e-01,
          3.9247e-01, -4.5473e-01,  6.5855e-02, -1.4232e-01, -2.1810e-01,
          9.2621e-02, -3.2060e-01, -1.4256e-01,  3.5293e-02, -1.0627e-01]],
       grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.4976, 0.4492, 0.0000, 0.1618, 0.0000,

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [11]:
seq_modules = nn.Sequential(#有序模块容器，数据可以按照定义顺序依次流经所有模块
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [ ]:
softmax = nn.Softmax(dim=1)#规定第一维，即每一行的值之和要为1
pred_probab = softmax(logits)#最终得到的 就是每个类别的概率

Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


//自增笔记📒：

框架会自动追踪定义的所有参数

- parameters():返回参数张量
- named_parameters():返回参数名+参数张量

In [13]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0032, -0.0294, -0.0289,  ...,  0.0320, -0.0321, -0.0204],
        [-0.0168,  0.0192, -0.0285,  ...,  0.0007, -0.0132, -0.0038]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([-0.0213, -0.0308], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0208, -0.0229,  0.0096,  ..., -0.0309,  0.0224,  0.0441],
        [ 0.0153, -0.0229,  0.0431,  ..., -0.0351, -0.0090,  0.0413]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | 

------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)
